In [8]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load your CSV
csv_path = "/content/wiki_hybrid_chunks_300.csv"
df = pd.read_csv(csv_path)
chunks = df["chunk_text"].tolist()

# Load TinyLLaMA model and tokenizer (adjust device if you want to use GPU)
tinyllama_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_id)
model = AutoModelForCausalLM.from_pretrained(tinyllama_model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Helper function to generate question for one chunk
def generate_question(chunk_text, max_new_tokens=40):
    prompt = f"""Based on the following text, generate one factual question.

Text:
{chunk_text[:500]}

Question:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        num_return_sequences=1,
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract question after "Question:"
    question = generated_text.split("Question:")[-1].strip().split("\n")[0]
    return question

# Generate questions for first 100 chunks (or change as needed)
generated_questions = []

for i, chunk in enumerate(chunks[:100]):
    try:
        question = generate_question(chunk)
        generated_questions.append({"chunk_id": df.iloc[i]["chunk_id"], "question": question})
        print(f"[{i+1}] {question}")
    except Exception as e:
        print(f"Error on chunk {i+1}: {e}")

# Save generated questions to CSV
questions_df = pd.DataFrame(generated_questions)
questions_df.to_csv("generated_questions_tinyllama.csv2", index=False)
print("Saved questions to generated_questions_tinyllama.csv2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[1] What is marieve herington's age as of 2020?
[2] What is the title of Marieve Herington's latest album, and when was it released?
[3] What word is used to describe honoka?
[4] What is the name of the actress who was born in Los Angeles and is a jazz singer, video game actress, and television actress?
[5] How many teams did robert brown play for during his minor league hockey career, and which NHL team did he shuffle between for several years?
[6] What was the career-high scoring total for the 1988-89 season for Hartford Whalers player, Tomas Plekanec?
[7] What are the details of the factual question regarding the career of J.J. Brown, including his playing time, the number of games he played, and the number of points he contributed in his
[8] How many games did the New York Rangers win during the regular season?
[9] What is the name of the organization that provides instruction for oilers hockey institute instructors, and what is the purpose of their training?
[10] What is the offic